
# Demo 06: Decision Tree

**Note**: This notebook is a "cleaned-up" version of the code written in class; it contains only minimal textual explanation. Please refer to the lessons for further information about what is done, and why.


Decision trees are a fundamental tool in machine learning, utilized for both classification and regression tasks. They operate by recursively splitting a dataset into subsets based on feature values, forming a tree-like structure where each internal node represents a decision based on a feature, and each leaf node represents an outcome or class label.

**Methods:**

1. **Splitting Criteria:**
   - **Gini Impurity:** Measures the frequency at which a randomly chosen element would be incorrectly classified. A lower Gini impurity indicates a better split. citeturn0search12
   - **Entropy and Information Gain:** Entropy quantifies the impurity or disorder of a dataset. Information gain calculates the reduction in entropy after a dataset is split on an attribute, guiding the selection of the best attribute for splitting. citeturn0search9

2. **Pruning:**
   - **Cost-Complexity Pruning (CCP):** Balances the complexity of the tree with its performance by removing branches that add little predictive power, thereby preventing overfitting. citeturn0search3

**Why These Methods Are Used:**

- **Gini Impurity and Entropy:** These metrics assess the quality of splits, ensuring that each node in the tree increases the homogeneity of the target variable within its subsets, leading to more accurate and generalizable models.
- **Pruning:** Pruning techniques like CCP help in simplifying the model, reducing overfitting, and enhancing the model's ability to generalize to unseen data.

**Results:**

When effectively implemented, decision trees offer:

- **Interpretability:** The tree structure provides a clear visualization of decision paths, making it easy to understand and interpret.
- **Flexibility:** Capable of handling both numerical and categorical data, decision trees are versatile across various types of datasets.
- **Efficiency:** They require relatively low computational resources for training and prediction, making them suitable for large datasets.

However, without proper techniques like pruning, decision trees can become overly complex and prone to overfitting, which can negatively impact their performance on new, unseen data. 

In [ ]:

from matplotlib import pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs

# Generate some toy data
X, y = make_blobs(n_samples=[10, 15, 25], n_features=2, centers=[[-1, -1], [0, 1], [1, 0]])
X, y


In [ ]:

from matplotlib.colors import ListedColormap

# Plot generated data
rgb_map = ListedColormap(('r', 'g', 'b'))
plt.grid(True)
plt.axhline(0.0, c='k')
plt.axvline(0.0, c='k')
plt.legend(*plt.scatter(X[:, 0], X[:, 1], c=y, cmap=rgb_map, edgecolors='w').legend_elements(), title='Classes:')
plt.title('$y(x)$')
plt.axis('square')
plt.axis((-4.0, 4.0, -4.0, 4.0))
plt.show()


In [ ]:

# Decision tree cost function criteria
# Gini impurity and Shannon entropy
criteria = {
    'gini': lambda p: 1.0 - np.sum(p ** 2),
    'entropy': lambda p: np.sum(-p * np.log2(np.maximum(p, 1e-100)))
}
ps = np.linspace(0.0, 1.0, 101)

# Plot cost function for Gini and Entropy
plt.grid(True)
for criterion, cost_function in criteria.items():
    cs = [cost_function(np.array([p, 1.0 - p])) for p in ps]
    plt.plot(ps, cs, '-', label=criterion)
plt.legend()
plt.title('Binary $C(p)$')
plt.show()


In [ ]:

# One-hot encoding for classes
classes = np.unique(y)
y_onehot = (y[:, np.newaxis] == classes)

# Check encoded classes
y_onehot


In [ ]:

# Class probabilities
p = y_onehot.mean(axis=0)
for criterion, cost_function in criteria.items():
    cost = cost_function(p)
    print(f'{criterion:9s} : {cost:.3f}')


In [ ]:

# Decision stump (one-level decision tree)
def determine_split(X, y_onehot, *, criterion='entropy'):
    '''Determine an optimal split feature and threshold from training data.'''
    cost_function = criteria[criterion]
    best_feature = best_threshold = None
    best_cost = cost_function(y_onehot.mean(axis=0))
    for feature in range(X.shape[1]):
        ordered = np.unique(X[:, feature])
        for threshold in (ordered[:-1] + ordered[1:]) / 2.0:
            subset = (X[:, feature] <= threshold)
            fraction = subset.mean()
            cost = (fraction * cost_function(y_onehot[subset, :].mean(axis=0)) +
                    (1.0 - fraction) * cost_function(y_onehot[~subset, :].mean(axis=0)))
            if cost < best_cost:
                best_feature, best_threshold, best_cost = feature, threshold, cost
    return best_feature, best_threshold

feature, threshold = determine_split(X, y_onehot)
feature, threshold


In [ ]:

# Plot the split point
for c in classes:
    plt.hist(X[y == c, feature], alpha=0.4, label=f'class {c}')
plt.axvline(threshold, c='k', ls=':', label='split')
plt.legend()
plt.show()


In [ ]:

# Predict probabilities using a simple decision stump
def predict_proba(X, y, X_test, *, criterion='entropy'):
    '''Trains a decision stump and predicts class probabilities for test data.'''
    classes = np.unique(y)
    y_onehot = (y[:, np.newaxis] == classes)
    feature, threshold = determine_split(X, y_onehot, criterion=criterion)
    subset = (X[:, feature] <= threshold)
    subset_test = (X_test[:, feature] <= threshold)
    y_proba = np.empty((X_test.shape[0], len(classes)))
    y_proba[subset_test, :] = y_onehot[subset, :].mean(axis=0)
    y_proba[~subset_test, :] = y_onehot[~subset, :].mean(axis=0)
    return y_proba

y_proba = predict_proba(X, y, X)
y_proba


In [ ]:

# Prediction function
def predict(*args, **kwargs):
    '''Trains a decision stump and predicts class labels for test data.'''
    return np.argmax(predict_proba(*args, **kwargs), axis=1)

y_hat = predict(X, y, X)
y_hat == y


In [ ]:

# Visualization of prediction regions for decision stump
xx1, xx2 = np.meshgrid(np.linspace(-4.0, 4.0, 81), np.linspace(-4.0, 4.0, 81))
XX = np.array([xx1.ravel(), xx2.ravel()]).T
yy = predict_proba(X, y, XX).reshape(xx1.shape + (3, ))

plt.grid(True)
plt.axhline(0.0, c='k')
plt.axvline(0.0, c='k')
plt.imshow(yy, extent=(-4.0, 4.0, -4.0, 4.0), origin='lower', interpolation='bilinear', alpha=0.5)
for z in range(yy.shape[2]):
    plt.contour(xx1, xx2, yy[:, :, z] / (1.0 - yy.min(axis=2)), [0.5], linewidths=0.5, linestyles='dashed')
plt.legend(*plt.scatter(X[:, 0], X[:, 1], c=y, cmap=rgb_map, edgecolors='w').legend_elements(), title='Classes:')
plt.title('$ŷ(x)$')
plt.axis('square')
plt.axis((-4.0, 4.0, -4.0, 4.0))
plt.show()


In [ ]:

# Using Scikit-learn DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

model = DecisionTreeClassifier(criterion='gini', max_depth=3, min_samples_split=5)
model.fit(X, y)

# Plot decision tree structure
plt.figure(figsize=(12, 6))
plot_tree(model, filled=True, label='root')
plt.show()


In [ ]:

# Probabilities and predictions using sklearn's DecisionTreeClassifier
y_proba = model.predict_proba(X)
y_hat = model.predict(X)
y_proba, y_hat
